In [2]:
#loading the data

import pandas as pd

customers=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\customers.csv")
orders=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\orders.csv")
items=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\order_items.csv")
products=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\products.csv")
shipments=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\shipments.csv")
sellers=pd.read_csv(r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\quickkart_dataset\sellers.csv")

In [3]:
#quality checks

tables={
"customers":customers,
"orders":orders,
"items":items,
"products":products,
"sellers":sellers,
"shipments":shipments
}

for name,df in tables.items():

    print("\n",name)

    print("Shape:",df.shape)

    print("\nNulls")
    print(df.isnull().sum())

    print("\nDuplicates")
    print(df.duplicated().sum())

    print("\nDatatypes")
    print(df.dtypes)


 customers
Shape: (25000, 5)

Nulls
customer_id    0
signup_date    0
city           0
state          0
segment        0
dtype: int64

Duplicates
0

Datatypes
customer_id    object
signup_date    object
city           object
state          object
segment        object
dtype: object

 orders
Shape: (100000, 7)

Nulls
order_id                     0
customer_id                  0
created_at                   0
status                       0
payment_method               0
promised_delivery_date       0
is_fast_delivery_eligible    0
dtype: int64

Duplicates
0

Datatypes
order_id                     object
customer_id                  object
created_at                   object
status                       object
payment_method               object
promised_delivery_date       object
is_fast_delivery_eligible      bool
dtype: object

 items
Shape: (169929, 8)

Nulls
order_item_id       0
order_id            0
product_id          0
seller_id           0
quantity            0
unit_price      

In [4]:
#fix datatypes

customers["signup_date"]=pd.to_datetime(customers["signup_date"])

orders["created_at"]=pd.to_datetime(
orders["created_at"]
)

orders["promised_delivery_date"]=pd.to_datetime(
orders["promised_delivery_date"]
)

shipments["shipped_at"]=pd.to_datetime(
shipments["shipped_at"]
)

shipments["delivered_at"]=pd.to_datetime(
shipments["delivered_at"]
)

In [5]:
#shipment nulls

shipments["delivered_at"]=(
shipments["delivered_at"]
)

customers.drop_duplicates(inplace=True)
orders.drop_duplicates(inplace=True)

In [6]:
#master dataset

df=(
orders
.merge(customers,on="customer_id")

.merge(items,on="order_id")

.merge(products,on="product_id")

.merge(sellers,on="seller_id")

.merge(shipments,on="order_id",how="left")
)

In [7]:
#Calculating the metrics

df["GMV"]=(
df["quantity"]
*
df["unit_price"]
)

df["Net_GMV"]=(
df["GMV"]
*
(1-df["discount_pct"])
)

df["Platform_Revenue"]=(
df["GMV"]
*
df["platform_fee_pct"]
)

df["month"]=(
df["created_at"]
.dt.to_period("M")
)

df["delay_days"]=(
df["delivered_at"]
-
df["promised_delivery_date"]
).dt.days

df["is_delayed"]=(
df["delivery_status"]
!="OnTime"
)

df["shipping_margin"]=(
df["Platform_Revenue"]
-
df["shipping_cost"]
)

In [8]:
#Answering questions
# Monthly GMV by city and category.

monthly_gmv=(
df
.groupby(
["month","city","category"]
)
.agg(
GMV=("GMV","sum"),

orders=("order_id","nunique")
)

.reset_index()
)

print(monthly_gmv)

        month       city        category         GMV  orders
0     2024-07  Ahmedabad           Books    110449.0      50
1     2024-07  Ahmedabad     Electronics   6231837.0      87
2     2024-07  Ahmedabad         Fashion    672491.0     101
3     2024-07  Ahmedabad         Grocery    117559.0      79
4     2024-07  Ahmedabad  Home & Kitchen   1341740.0      69
...       ...        ...             ...         ...     ...
1075  2025-12       Pune           Books    162273.0      79
1076  2025-12       Pune     Electronics  12332499.0     190
1077  2025-12       Pune         Fashion   1189547.0     189
1078  2025-12       Pune         Grocery    228416.0     141
1079  2025-12       Pune  Home & Kitchen   2114634.0     118

[1080 rows x 5 columns]


In [9]:
#Monthly Orders + Active Customers

orders_month=(
df
.groupby("month")
.agg(
orders=("order_id","nunique"),

customers=("customer_id","nunique")
)

.reset_index()
)

print(orders_month)

      month  orders  customers
0   2024-07    5634       4947
1   2024-08    5551       4890
2   2024-09    5595       4930
3   2024-10    5791       5102
4   2024-11    5504       4849
5   2024-12    5719       4952
6   2025-01    5604       4898
7   2025-02    5163       4581
8   2025-03    5627       4964
9   2025-04    5515       4866
10  2025-05    5536       4850
11  2025-06    5463       4838
12  2025-07    5580       4912
13  2025-08    5659       4950
14  2025-09    5380       4743
15  2025-10    5722       5006
16  2025-11    5491       4876
17  2025-12    5466       4836


In [10]:
#repeat purchase rate

delivered=(
df[
df["status"]=="Delivered"
]

[[
"customer_id",
"order_id",
"created_at"
]]

.drop_duplicates()
)

delivered=(
delivered
.sort_values(
[
"customer_id",
"created_at"
]
)
)

delivered["order_rank"]=(
delivered
.groupby(
"customer_id"
)
.cumcount()+1
)

repeat=(
delivered
.groupby(
delivered["created_at"]
.dt.to_period("M")
)

.apply(
lambda x:
(
x["order_rank"]>=2
).mean()
)

.reset_index(
name="repeat_rate"
)
)

print(delivered)

       customer_id   order_id created_at  order_rank
136353      C00001  ORD080280 2025-09-14           1
8731        C00002  ORD005109 2024-07-29           1
11338       C00002  ORD006629 2024-08-06           2
88902       C00002  ORD052350 2025-04-12           3
108782      C00002  ORD064088 2025-06-16           4
...            ...        ...        ...         ...
6420        C25000  ORD003758 2024-07-21           2
42466       C25000  ORD025016 2024-11-14           3
44844       C25000  ORD026405 2024-11-22           4
106641      C25000  ORD062807 2025-06-09           5
152984      C25000  ORD090075 2025-11-06           6

[82069 rows x 4 columns]


In [11]:
#delayed orders share

delay_share=(
df
.groupby(
[
"ship_to_city",
"carrier"
]
)

["is_delayed"]

.mean()

.reset_index()
)

print(delay_share)

   ship_to_city    carrier  is_delayed
0     Ahmedabad   BlueDart    0.317823
1     Ahmedabad  Delhivery    0.326768
2     Ahmedabad      Ekart    0.303419
3     Bangalore   BlueDart    0.314002
4     Bangalore  Delhivery    0.316089
5     Bangalore      Ekart    0.326950
6     Bangalore    InHouse    0.127537
7    Chandigarh   BlueDart    0.313454
8    Chandigarh  Delhivery    0.313793
9    Chandigarh      Ekart    0.308668
10      Chennai   BlueDart    0.325215
11      Chennai  Delhivery    0.321845
12      Chennai      Ekart    0.313730
13        Delhi   BlueDart    0.326606
14        Delhi  Delhivery    0.309034
15        Delhi      Ekart    0.321620
16        Delhi    InHouse    0.139187
17    Hyderabad   BlueDart    0.306258
18    Hyderabad  Delhivery    0.318020
19    Hyderabad      Ekart    0.333882
20    Hyderabad    InHouse    0.131442
21       Jaipur   BlueDart    0.325484
22       Jaipur  Delhivery    0.865385
23       Jaipur      Ekart    0.325138
24        Kochi   BlueDar

In [12]:
exports={

r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\clean_master_data.csv":df,

r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\monthly_gmv.csv":monthly_gmv,

r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\monthly_orders.csv":orders_month,

r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\repeat_rate.csv":repeat,

r"C:\Users\wasee\OneDrive - Phoenix\Desktop\python code\delay_share.csv":delay_share

}

for file,data in exports.items():

    data.to_csv(
        file,
        index=False
    )

print("Export complete")